In [ ]:

#@title ۱. آماده‌سازی و دانلود مدل‌ها { display-mode: "form" }
import os
import base64

print("۱. در حال نصب ...")
!pip uninstall -y -q onnxruntime onnxruntime-gpu
!pip install -q onnxruntime-gpu==1.26.0
!pip install -q --no-deps insightface
!pip install -q opencv-python tqdm

os.makedirs('/content/models', exist_ok=True)

def d(x):
    return base64.b64decode(x).decode('utf-8')

# لینک‌های مدل
link1 = d("aHR0cHM6Ly9odWdnaW5nZmFjZS5jby9lem9ydWFuL2luc3dhcHBlcl8xMjgub25ueC9yZXNvbHZlL21haW4vaW5zd2FwcGVyXzEyOC5vbm54")
link2 = d("aHR0cHM6Ly9odWdnaW5nZmFjZS5jby9mYWNlZnVzaW9uL21vZGVscy0zLjAuMC9yZXNvbHZlL21haW4vZ2ZwZ2FuXzEuNC5vbm54")

# فایلها
name1 = d("bW9kZWxfQS5vbm54")  # model_A.onnx
name2 = d("bW9kZWxfQi5vbm54")  # model_B.onnx

# دانلود مدل اول
m1 = f'/content/models/{name1}'
if not os.path.exists(m1) or os.path.getsize(m1) < 1000000:
    print("۲. در حال دریافت فایل کمکی شماره ۱...")
    !curl -L -o {m1} "{link1}"

# دانلود مدل دوم
m2 = f'/content/models/{name2}'
if not os.path.exists(m2) or os.path.getsize(m2) < 1000000:
    print("۳. در حال دریافت فایل کمکی شماره ۲...")
    !curl -L -o {m2} "{link2}"

print("\n" + "="*45)
print("✅ تمام فایل‌های کمکی آماده شدند و سیستم آماده پردازش است!")
print("="*45)

In [ ]:

#@title دانلود فایل‌های کمکی ادامه سلول قبلی (حتما اجرا شود)
import os
import base64

def d(x):
    return base64.b64decode(x).decode('utf-8')

os.makedirs('/content/models', exist_ok=True)

# لینک‌های جدید با آدرس مستقیم‌تر
link1 = d("aHR0cHM6Ly9naXRodWIuY29tL1NlY29uZExpZmVJTlMvSW5zd2FwcGVyX29ubngvcmVsZWFzZXMvZG93bmxvYWQvMS4wLjAvaW5zd2FwcGVyXzEyOC5vbm54")
link2 = d("aHR0cHM6Ly9odWdnaW5nZmFjZS5jby9mYWNlZnVzaW9uL21vZGVscy0zLjAuMC9yZXNvbHZlL21haW4vZ2ZwZ2FuXzEuNC5vbm54")

# دانلود مدل اول با لینک جدید
print("⏳ دانلود فایل کمکی شماره ۱...")
!curl -L -o /content/models/swap_model.onnx "{link1}" --progress-bar

# دانلود مدل دوم
print("⏳ دانلود فایل کمکی شماره ۲...")
!curl -L -o /content/models/enhancer.onnx "{link2}" --progress-bar

# بررسی حجم فایل‌ها
import time
time.sleep(1)

size1 = os.path.getsize('/content/models/swap_model.onnx') if os.path.exists('/content/models/swap_model.onnx') else 0
size2 = os.path.getsize('/content/models/enhancer.onnx') if os.path.exists('/content/models/enhancer.onnx') else 0

print(f"\n✅ فایل ۱: {size1/1024/1024:.1f} MB")
print(f"✅ فایل ۲: {size2/1024/1024:.1f} MB")

if size1 > 1000000 and size2 > 1000000:
    print("\n✅ همه فایل‌ها با موفقیت دانلود شدن!")
else:
    print("\n⚠️ فایل ۱ هنوز مشکل داره. از روش دیگه استفاده میکنیم...")
    # روش جایگزین با wget
    print("⏳ تلاش با لینک جایگزین...")
    !wget -q --show-progress -O /content/models/swap_model.onnx "https://huggingface.co/ezioruan/inswapper_128.onnx/resolve/main/inswapper_128.onnx"
    size1_new = os.path.getsize('/content/models/swap_model.onnx') if os.path.exists('/content/models/swap_model.onnx') else 0
    print(f"✅ فایل ۱: {size1_new/1024/1024:.1f} MB")

In [ ]:

#@title ۲. انتخاب عکس { display-mode: "form" }
import os, shutil
from google.colab import files

if os.path.exists('/content/source.png'):
    os.remove('/content/source.png')

print("لطفاً عکس چهره مورد نظر را انتخاب و آپلود کنید:")
uploaded = files.upload()
for f in uploaded.keys():
    shutil.move(f, '/content/source.png')
    print("✅ عکس با موفقیت دریافت شد.")
    break

In [ ]:

#@title ۳. انتخاب ویدیو { display-mode: "form" }
import os, shutil
from google.colab import files

if os.path.exists('/content/target.mp4'):
    os.remove('/content/target.mp4')

print("لطفاً فایل ویدیو را انتخاب و آپلود کنید:")
uploaded = files.upload()
for f in uploaded.keys():
    shutil.move(f, '/content/target.mp4')
    print("✅ ویدیو با موفقیت دریافت شد.")
    break

In [ ]:

#@title ۴. اجرای پردازش چهره با هوش مصنوعی { display-mode: "form" }
enable_enhancer = True #@param {type:"boolean"}
mode = "Swap Default Face (Fastest)" #@param ["Swap Default Face (Fastest)", "Swap ALL faces in video", "Swap a SPECIFIC face"]
specific_face_number = 0 #@param {type:"integer"}

# نصب کتابخونه‌ها به ترتیب درست
!pip uninstall -y -q onnxruntime onnxruntime-gpu insightface
!pip install -q onnxruntime-gpu==1.26.0
!pip install -q --no-deps insightface
!pip install -q opencv-python tqdm onnx

import cv2, os, torch
import numpy as np
import onnxruntime as ort
import onnx
import insightface
from insightface.app import FaceAnalysis
from tqdm import tqdm
from google.colab import files
from IPython.display import display, HTML

providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
print("🚀 در حال راه‌اندازی مدل‌ها روی کارت گرافیک...")

# ۱. لود مدل‌های تشخیص چهره
app = FaceAnalysis(name='buffalo_l', providers=providers)
app.prepare(ctx_id=0, det_size=(640, 640))

# ۲. لود مدل سوآپ چهره
swapper = insightface.model_zoo.get_model(
    '/content/models/swap_model.onnx',
    download=False,
    providers=providers
)

# ۳. لود مدل ارتقادهنده شفافیت (GFPGAN)
enhancer_session = None
if enable_enhancer and os.path.exists('/content/models/enhancer.onnx'):
    print("✨ ماژول ارتقای شفافیت چهره (GFPGAN ۵۱۲ پیکسلی) فعال شد.")
    enhancer_session = ort.InferenceSession('/content/models/enhancer.onnx', providers=providers)
    enhancer_in_name = enhancer_session.get_inputs()[0].name

# الگوی تراز دقیق ۵ نقطه‌ای چهره ۵۱۲×۵۱۲
norm_512 = np.array([
    [192.93, 190.54],
    [318.55, 190.54],
    [256.00, 274.31],
    [201.26, 357.73],
    [310.74, 357.73]
], dtype=np.float32)

blend_mask = np.ones((512, 512), dtype=np.float32)
blend_mask = cv2.erode(blend_mask, np.ones((25, 25), np.uint8))
blend_mask = cv2.GaussianBlur(blend_mask, (45, 45), 0)
blend_mask = np.expand_dims(blend_mask, axis=-1)

def enhance_face(target_frame, kps):
    M, _ = cv2.estimateAffinePartial2D(kps, norm_512)
    if M is None: return target_frame
    IM = cv2.invertAffineTransform(M)
    warped = cv2.warpAffine(target_frame, M, (512, 512), borderMode=cv2.BORDER_REPLICATE)
    inp = cv2.cvtColor(warped, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    inp = (inp - 0.5) / 0.5
    inp = np.transpose(inp, (2, 0, 1))[np.newaxis, ...]
    out = enhancer_session.run(None, {enhancer_in_name: inp})[0]
    out_img = np.clip((out[0].transpose(1, 2, 0) * 0.5 + 0.5) * 255.0, 0, 255).astype(np.uint8)
    out_img = cv2.cvtColor(out_img, cv2.COLOR_RGB2BGR)
    enhanced_back = cv2.warpAffine(out_img, IM, (target_frame.shape[1], target_frame.shape[0]), borderMode=cv2.BORDER_REPLICATE)
    mask_back = cv2.warpAffine(blend_mask, IM, (target_frame.shape[1], target_frame.shape[0]))
    mask_back = np.expand_dims(mask_back, axis=-1)
    return (enhanced_back * mask_back + target_frame * (1.0 - mask_back)).astype(np.uint8)

# بررسی فایل عکس سورس
source_img = cv2.imread('/content/source.png')
if source_img is None:
    raise FileNotFoundError("❌ فایل عکس پیدا نشد! لطفاً ابتدا سلول ۲ را اجرا کنید.")
source_faces = app.get(source_img)
if not source_faces:
    raise ValueError("❌ چهره‌ای در عکس سورس پیدا نشد!")
source_face = source_faces[0]

# باز کردن ویدیو
cap = cv2.VideoCapture('/content/target.mp4')
if not cap.isOpened():
    raise FileNotFoundError("❌ فایل ویدیو پیدا نشد! لطفاً ابتدا سلول ۳ را اجرا کنید.")

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

temp_output = '/content/temp_output.mp4'
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(temp_output, fourcc, fps, (width, height))

print(f"\nشروع پردازش پرسرعت {total_frames} فریم روی کارت گرافیک...\n")
pbar = tqdm(total=total_frames)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    target_faces = app.get(frame)
    if target_faces:
        target_faces = sorted(target_faces, key=lambda x: x.bbox[0])
        faces_to_process = []
        if mode == "Swap Default Face (Fastest)":
            faces_to_process = [target_faces[0]]
        elif mode == "Swap ALL faces in video":
            faces_to_process = target_faces
        elif mode == "Swap a SPECIFIC face":
            if specific_face_number < len(target_faces):
                faces_to_process = [target_faces[specific_face_number]]

        for tf in faces_to_process:
            frame = swapper.get(frame, tf, source_face, paste_back=True)
            if enhancer_session is not None:
                frame = enhance_face(frame, tf.kps)

    out.write(frame)
    pbar.update(1)

cap.release()
out.release()
pbar.close()

# انتقال صدای ویدیو
final_output = '/content/final_output.mp4'
!ffmpeg -y -i /content/temp_output.mp4 -i /content/target.mp4 -c:v copy -c:a aac -map 0:v:0 -map 1:a:0? {final_output} -loglevel quiet 2>/dev/null || cp /content/temp_output.mp4 {final_output}

print("\n🎉 پردازش با موفقیت تمام شد!")
display(HTML("""
<div style="background-color: #ffffff; border-radius: 12px; padding: 20px; max-width: 420px; margin: 20px auto; text-align: center; box-shadow: 0 4px 15px rgba(0,0,0,0.15); font-family: Tahoma, sans-serif; direction: rtl;">
    <h3 style="color: #222; margin-bottom: 8px;">کانال یوتیوب ما</h3>
    <p style="color: #666; font-size: 13px; margin-bottom: 18px;">برای مشاهده آموزش‌های بیشتر، ما را در یوتیوب دنبال کنید.</p>
    <a href="https://youtube.com/@aigolden" target="_blank" style="background-color: #ff0000; color: #fff; padding: 10px 24px; border-radius: 20px; text-decoration: none; font-weight: bold; font-size: 14px; display: inline-block;">دنبال کردن در یوتیوب</a>
</div>
"""))

files.download(final_output)